# Step 0
#### Install dependencies and import libraries

In [ ]:
import subprocess
import pkg_resources

def install_missing(requirements_file="requirements.txt"):
    # Legge il file
    with open(requirements_file, "r") as f:
        required = f.read().splitlines()

    # Pacchetti già installati
    installed = {pkg.key for pkg in pkg_resources.working_set}

    # Filtra quelli mancanti
    missing = []
    for req in required:
        pkg_name = req.strip().split("==")[0]
        if pkg_name.lower() not in installed:
            missing.append(req)

    if not missing:
        print("Tutti i pacchetti sono già installati.")
        return

    print("Installazione pacchetti mancanti:", missing)

    # Installa i pacchetti mancanti
    subprocess.check_call(["pip", "install"] + missing)

install_missing()

In [ ]:
# All imports
import torch
from datasets import load_dataset, Dataset, load_from_disk
from transformers import pipeline, AutoTokenizer, GenerationConfig, AutoModelForCausalLM
from tqdm.auto import tqdm
from transformers.pipelines.pt_utils import KeyDataset
from trl import SFTTrainer, SFTConfig
import evaluate
import math
import random

<a target="_blank" href="https://colab.research.google.com/github/WholeNow/KnowledgeDistillator/blob/main/Project.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Step 1
#### Create the dataset using as labels the prediction of tinyLlama-1.1-1T on the training set of the original dataset

In [ ]:


# 1. Inizializzazione Teacher ottimizzata per T4
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# La pipeline gestisce il batching in modo efficiente
generator = pipeline(
    "text-generation",
    model=model_id,
    dtype=torch.float16, # Obbligatorio per T4 (no bfloat16)
    device_map="auto",
    batch_size=16 # Ottimizzato per 16GB VRAM
)

# Funzione per formattare il prompt secondo il template di TinyLlama
def format_prompt(dialogue):
    messages = [
        {"role": "system", "content": "You are a highly accurate summarization assistant. Provide a concise summary of the following conversation."},
        {"role": "user", "content": f"Summarize this dialogue:\n\n{dialogue}"}
    ]
    # Applica il ChatML template nativo del modello
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


# Datasets
# knkarthick/samsum
# m-a-p/CodeFeedback-Filtered-Instruction

# 2. Caricamento e preparazione dataset
dataset = load_dataset("knkarthick/samsum", split="train[:500]") # Per test usa split="train[:500]"

print("Colonne del dataset:", dataset.column_names)

prompts = [format_prompt(dialogue) for dialogue in dataset["dialogue"]]
prompt_dataset = Dataset.from_dict({"prompt": prompts})

print("Inizio generazione pseudo-labels...")
teacher_summaries = []


gen_config = GenerationConfig(
    max_new_tokens=128,
    do_sample=False,
    return_full_text=False,
    max_length=None # Per evitare warning, ma non è usato se max_new_tokens è specificato
)

# tqdm genera la barra di avanzamento in tempo reale
# max_length=None sopprime il warning
for out in tqdm(
    generator(
        KeyDataset(prompt_dataset, "prompt"),
        generation_config=gen_config
    ),
    total=len(prompts),
    desc="Distillazione SamSum"
):
    teacher_summaries.append(out[0]['generated_text'].strip())

# 4. Aggiunta delle pseudo-label al dataset e salvataggio
distilled_dataset = dataset.add_column("teacher_summary", teacher_summaries)
distilled_dataset.save_to_disk("./samsum_distilled_tinyllama")
print("Dataset salvato con successo.")

# Step 2
#### Train a student model (SmolLM-360M-Instruct) on the dataset created in Step 1

In [ ]:

# 1. Inizializzazione
model_id = "HuggingFaceTB/SmolLM-135M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- FIX: Iniezione forzata del ChatML template ---
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "<|im_start|>{{ message['role'] }}\n"
        "{{ message['content'] }}<|im_end|>\n"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "<|im_start|>assistant\n"
        "{% endif %}"
    )

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float32, # Obbligatorio per T4 (no bfloat16)
    device_map="auto"
)

dataset = load_from_disk("./samsum_distilled_tinyllama")

# 2. Mappatura nel formato nativo Prompt-Completion
def format_to_prompt_completion(example):
    # La chiave qui è non usare apply_chat_template per il training,
    # ma strutturare un dizionario "messages" standard che trl sa parsare nativamente.
    messages = [
        {"role": "system", "content": "You are a highly accurate summarization assistant. Provide a concise summary of the following conversation."},
        {"role": "user", "content": f"Summarize this dialogue:\n\n{example['dialogue']}"},
        {"role": "assistant", "content": example['teacher_summary']}
    ]
    
    # Restituiamo direttamente la lista dei messaggi.
    # Il tokenizer e SFTTrainer si occuperanno internamente del template e del masking.
    return {"messages": messages}

# Applica la mappatura e sopprime le vecchie colonne per non creare conflitti nel data loader
pc_dataset = dataset.map(format_to_prompt_completion, remove_columns=dataset.column_names)

# 3. Configurazione con SFTConfig (sostituisce TrainingArguments)
training_args = SFTConfig(
    output_dir="./smollm_distilled_samsum",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    logging_steps=10,
    num_train_epochs=3,
    fp16=True,
    optim="adamw_torch_fused",
    report_to="none",
    max_length=1024,
    completion_only_loss=False, # Esegue il masking della loss dinamicamente sul 'prompt'
)

# 4. Esecuzione
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=pc_dataset,
    processing_class=tokenizer, # Sostituisce il passaggio esplicito del tokenizer
)

print("Avvio addestramento...")
trainer.train()

# Salvataggio
trainer.save_model("./smollm_distilled_samsum_final")
tokenizer.save_pretrained("./smollm_distilled_samsum_final")

# Step 3
#### Evaluate the student model on the original test set, comparing it with the teacher model (tinyLlama-1.1-1T) and with a baseline (SmolLM-360M-Instruct fine-tuned on the original dataset)

In [ ]:
# 1. Setup metriche e hardware
device = "cuda" if torch.cuda.is_available() else "cpu"
rouge_metric = evaluate.load("rouge")
bert_metric = evaluate.load("bertscore")

print(f"Device per valutazione: {device}")

# Caricamento del dataset di test originale
test_dataset = load_dataset("knkarthick/samsum", split="test[:100]") # Per velocizzare i test: split="test[:100]"

# 2. Funzione di generazione per la valutazione
def evaluate_model(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # --- FIX: Iniezione forzata del ChatML template ---
    if tokenizer.chat_template is None:
        tokenizer.chat_template = (
            "{% for message in messages %}"
            "<|im_start|>{{ message['role'] }}\n"
            "{{ message['content'] }}<|im_end|>\n"
            "{% endfor %}"
            "{% if add_generation_prompt %}"
            "<|im_start|>assistant\n"
            "{% endif %}"
        )

    # Caricamento in FP32 per stabilità su T4, l'inferenza può usare fp16 se necessario
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float32).to(device)
    model.eval()
    
    predictions = []
    references = []
    total_loss = 0.0 # Nuovo accumulatore
    
    print(label := f"Valutazione del modello: {model_path}")
    for sample in tqdm(test_dataset, desc=label):
        messages = [
            {"role": "system", "content": "You are a highly accurate summarization assistant. Provide a concise summary of the following conversation."},
            {"role": "user", "content": f"Summarize this dialogue:\n\n{sample['dialogue']}"}
        ]
        
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
        
        # --- NUOVO BLOCCO: Calcolo Loss e Perplexity ---
        # Creiamo un tensore combinato [prompt + summary_reale] per calcolare la loss
        full_text = prompt + sample['summary'] + tokenizer.eos_token
        full_inputs = tokenizer(full_text, return_tensors="pt").to(device)
        
        with torch.no_grad():
            # Passando le "labels", il CausalLM calcola in automatico la Cross-Entropy Loss
            loss_output = model(**full_inputs, labels=full_inputs["input_ids"])
            total_loss += loss_output.loss.item()
            
            # --- Generazione standard per ROUGE/BERTScore ---
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )
            
        generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        gen_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        
        predictions.append(gen_text)
        references.append(sample['summary'])
        
    # Calcolo metriche
    rouge_results = rouge_metric.compute(predictions=predictions, references=references)
    bert_results = bert_metric.compute(predictions=predictions, references=references, lang="en")
    
    # Aggregazione finale
    mean_bert_f1 = sum(bert_results['f1']) / len(bert_results['f1'])
    avg_loss = total_loss / len(test_dataset)
    perplexity = math.exp(avg_loss) # Formula matematica della PPL
    
    return {
        "Perplexity": perplexity,
        "ROUGE-1": rouge_results['rouge1'],
        "ROUGE-2": rouge_results['rouge2'],
        "ROUGE-L": rouge_results['rougeL'],
        "BERTScore-F1": mean_bert_f1
    }

# 3. Esecuzione dei confronti
metrics_distilled = evaluate_model("./smollm_distilled_samsum_final")
metrics_baseline = evaluate_model("HuggingFaceTB/SmolLM-135M")
metrics_teacher = evaluate_model("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

# 4. Stampa comparativa dei risultati
print("\n=== RISULTATI COMPARATIVI ===")
for name, metrics in [("Teacher (TinyLlama)", metrics_teacher), 
                      ("Student Baseline", metrics_baseline), 
                      ("Student Distilled", metrics_distilled)]:
    print(f"\n{name}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")




## Esempio Visivo

In [ ]:
# 1. Caricamento Modello Distillato
model_path = "./smollm_distilled_samsum_final"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")

# 2. Caricamento Test Set e Selezione Random
dataset = load_dataset("knkarthick/samsum", split="test")
samples = random.sample(list(dataset), 3)

print("=== ISPEZIONE QUALITATIVA DEGLI OUTPUT ===\n")

for i, sample in enumerate(samples, 1):
    messages = [
        {"role": "system", "content": "You are a highly accurate summarization assistant. Provide a concise summary of the following conversation."},
        {"role": "user", "content": f"Summarize this dialogue:\n\n{sample['dialogue']}"}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=128, 
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
        
    gen_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    
    print(f"--- SAMPLE {i} ---")
    print(f"DIALOGO ORIGINALE:\n{sample['dialogue'].strip()}\n")
    print(f"TARGET IDEALE (Human):\n{sample['summary'].strip()}\n")
    print(f"GENERAZIONE STUDENT DISTILLATO:\n{gen_text.strip()}\n")
    print("="*50 + "\n")